# Sleep–Frailty: small free-runtime portability check

This notebook checks the **existing** fitted molecular model on ten public historical DNA examples. It does not train a new model or evaluate the reserved study. A pass is a numerical portability check, not biological validation.

Start with a standard CPU runtime. No paid runtime, subscription, Drive mount, API key, or broad account permission is needed by these cells. Colab availability and session limits vary; see https://research.google.com/colaboratory/faq.html.

Nothing has been uploaded by preparing this notebook. Only the exact reviewed files should be uploaded after the destination/account is confirmed.


In [ ]:
import sys, os, shutil, platform, json
sys.dont_write_bytecode = True
info = {"platform": platform.platform(), "python": sys.version, "disk_free_GiB": shutil.disk_usage(".").free / 2**30}
try:
    info["RAM_GiB"] = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
except (AttributeError, ValueError):
    pass
print(json.dumps(info, indent=2))


## Reviewed small input set

- `WTC11_author_test1_val2.safetensors` (3476012 bytes)
- `cell_matched_effect_network.py` (2355 bytes)
- `fixed_channel_projections.npy` (24704 bytes)
- `frozen_reader_none_42_epoch3.safetensors` (78332 bytes)
- `legnet_model.py` (6400 bytes)
- `native_probe_configuration.json` (15789 bytes)
- `portable_native_probe.py` (7978 bytes)
- `public_DNA_probe_inputs.npz` (8076 bytes)

Total: 3619646 bytes. The files contain public DNA/model inputs, existing model weights and numerical reference outputs. Experimental outcome labels were removed from the DNA fixture. There are no patient tables, manuscripts, secrets, account credentials or reserved-study outcomes.


In [ ]:
import importlib.util, subprocess
if importlib.util.find_spec("safetensors") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "safetensors"])
import torch, numpy
print({"torch": torch.__version__, "numpy": numpy.__version__, "probe_device": "CPU float64"})


In [ ]:
from google.colab import files
# Select ONLY the eight reviewed input files listed above.
uploaded = files.upload()
print("Received file names:", sorted(uploaded))


In [ ]:
from pathlib import Path
import hashlib
expected = {'WTC11_author_test1_val2.safetensors': '3f297fc40c0ff146589e0158db556988a51c1d779eb7ae11c3773269c18aa62a', 'cell_matched_effect_network.py': '220205ff5e68e4ae69460bee42810db98dfdc31492dfd70a5eac05321318765e', 'fixed_channel_projections.npy': '7bd9272afa06e0608fed98038c9b0d50d59a0a61c4ce49ac4cf54df233665d54', 'frozen_reader_none_42_epoch3.safetensors': '0464f3fdee517506d532a70626dff874e850608b87d5b3f8c48c5850bd309324', 'legnet_model.py': '8a5650b788662db07fda0e82566697b0aa1b81a1e6ff7a53448f138d00c40975', 'native_probe_configuration.json': 'db2fd66e71996579497cbce7586b22a6c1897e8780e1a611314eb0b6edf0b00d', 'portable_native_probe.py': '63ebfb530837831949df90a604c5bc76d66d9c37a39c3b56687328d08895f8a1', 'public_DNA_probe_inputs.npz': '8f7686019032bb89d924d5c19a0c25539a392b367c3b6aa9853f11d01705bc43'}
for filename, digest in expected.items():
    path = Path(filename)
    assert path.is_file(), f"Missing reviewed file: {filename}"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Input changed: {filename}"
print("All eight reviewed file hashes match.")


In [ ]:
spec = importlib.util.spec_from_file_location("reviewed_portability_probe", "portable_native_probe.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
result = module.probe(Path.cwd())
print(json.dumps(result, indent=2))


## Meaning of the result

A pass means the original model reproduces its saved reference predictions within the stated numerical tolerance and preserves the expected behavior when the two DNA versions are exchanged. It establishes neither improved accuracy nor genetic causation, clinical prediction, independent genetic-score validation or successful AlphaGenome-score integration.

This is **not** the full cloud training pipeline. A separate controlled execution plan is needed before moving the full comparison. A Mac GPU random stream cannot be claimed to continue exactly on different hardware. The original Mac checkpoint and analyses remain preserved.


In [ ]:
# Optional: download the small numerical-check result after reviewing it.
# files.download("portable_probe_result.json")
